# Computational Biophysics: From Neurons to Networks
## A Book-Style Tutorial on Neural Simulations, TFNE, and Optimization

---


## 0. Opening: Neuronal Equations

Neural modeling spans multiple scales of abstraction.

### Comparison of Model Families

| Model | State Variables | Units | Timescale | Biological Detail | Cost | Best Use | Failure Mode |
|:---|:---|:---|:---|:---|:---|:---|:---|
| **Point-Process** | Spike times | Binary/None | ms | Low | Very Low | Population statistics | Missing subthreshold dynamics |
| **Rate-Coded** | Firing rate | Hz | 10-100ms | Low | Low | Large-scale connectivity | No precise spike timing |
| **Discrete-Time** | Voltage (V) | Unitless/mV | dt | Low | Low | Fast prototyping | Sensitive to dt choice |
| **LIF** | V | mV | ms | Low-Med | Low | Information theory | No spike shape/bursting |
| **QIF/EIF** | V | mV | ms | Med | Low | Bifurcation analysis | Single-compartment only |
| **Izhikevich** | V, u | Unitless* | ms | Med | Low | Diverse dynamics (bursting) | Phenomenological parameters |
| **HH (Full)** | V, m, n, h | mV | 0.01ms | High | High | Channel dynamics | Parameter explosion |
| **Cable Theory** | V(x,t) | mV | <0.01ms | Very High | Very High | Dendritic computation | Computationally prohibitive |
| **TFNE** | phi, V_m | mV, V | ms | Med | Med | Hybrid forward-field (LFP) | Unverified biological truth |

*Note: Izhikevich equations are typically scaled such that V is roughly in mV but the underlying math is dimensionless. Use 'Izhikevich current units' for input.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import optax
import sys
import os
from typing import NamedTuple, Callable

# Configuration
SMOKE_MODE = True

if SMOKE_MODE:
    DURATION_MS = 200.0
    N_POP = 20
else:
    DURATION_MS = 1000.0
    N_POP = 100

DT_MS = 0.1
STEPS = int(DURATION_MS / DT_MS)
T_MS = jnp.arange(STEPS) * DT_MS

print(f"Setup: SMOKE_MODE={SMOKE_MODE}, DURATION={DURATION_MS}ms")


## 1. JAX: The Foundation


In [ ]:
@jax.jit
def jax_check(x):
    return jnp.sin(x)**2 + jnp.cos(x)**2
print(f"JAX OK: {jax_check(1.0)}")


## 2. Simulate a Neuron (Izhikevich)

**Critical Scientific Fix:** The canonical Izhikevich equation is ms-scaled.


In [ ]:
class IzhikevichState(NamedTuple):
    v: jnp.ndarray
    u: jnp.ndarray

class IzhikevichParams(NamedTuple):
    a: jnp.ndarray = jnp.array(0.02)
    b: jnp.ndarray = jnp.array(0.2)
    c: jnp.ndarray = jnp.array(-65.0)
    d: jnp.ndarray = jnp.array(8.0)

def izh_step(state, params, I, dt):
    v, u = state.v, state.u
    dv = 0.04 * v**2 + 5 * v + 140 - u + I
    du = params.a * (params.b * v - u)
    v_new = v + dt * dv
    u_new = u + dt * du
    spike = v_new >= 30.0
    v_next = jnp.where(spike, params.c, v_new)
    u_next = jnp.where(spike, u_new + params.d, u_new)
    return IzhikevichState(v_next, u_next), (v, spike)

def run_sim(I_val=10.0):
    params = IzhikevichParams()
    init = IzhikevichState(jnp.array(params.c), jnp.array(params.b * params.c))
    def body(s, t):
        I = jnp.where((t > 50) & (t < 150), I_val, 0.0)
        return izh_step(s, params, I, DT_MS)
    _, (v, spk) = jax.lax.scan(body, init, T_MS)
    return v, spk

v, spk = run_sim()
plt.plot(T_MS, v)
plt.show()


## 3. Simulate More Neurons (vmap)


In [ ]:
def run_many(n=10):
    params = IzhikevichParams(jnp.full(n, 0.02), jnp.full(n, 0.2), jnp.full(n, -65.0), jnp.full(n, 8.0))
    init = IzhikevichState(params.c, params.b * params.c)
    def body(s, t):
        I = jnp.where((t > 50) & (t < 150), 10.0, 0.0)
        return jax.vmap(izh_step, in_axes=(0,0,0,None))(s, params, I, DT_MS)
    _, (v, spk) = jax.lax.scan(body, init, T_MS)
    return v, spk

v_many, spk_many = run_many(N_POP)
plt.imshow(spk_many.T, aspect='auto')
plt.show()


## 4. Cell Types: E, PV, SST, VIP


In [ ]:
CELL_REGISTRY = {
    'E': IzhikevichParams(0.02, 0.2, -65.0, 8.0),
    'PV': IzhikevichParams(0.1, 0.2, -65.0, 2.0),
    'SST': IzhikevichParams(0.02, 0.25, -65.0, 2.0),
    'VIP': IzhikevichParams(0.02, 0.2, -65.0, 2.0)
}
print("Registry loaded.")


## 5. Populations


In [ ]:
def make_pop(n, cell_type):
    p = CELL_REGISTRY[cell_type]
    return IzhikevichParams(jnp.full(n, p.a), jnp.full(n, p.b), jnp.full(n, p.c), jnp.full(n, p.d))
pop_e = make_pop(N_POP, 'E')


## 6. Cortical Column (make_cortex_network)


In [ ]:
from jbiophysic.networks import make_cortex_network
net = make_cortex_network(N=N_POP*2, XYZ_mm=(0.5, 0.5, 1.5))
print(f"Network with {len(net['types'])} neurons created.")


## 7. Multi-Area


In [ ]:
def delay_ms(dist_mm, v=1.0, lat=1.5):
    return (dist_mm / 1000.0 / v) * 1000.0 + lat
print(f"Delay: {delay_ms(5.0)} ms")


## 8. TFNE (Forward Field)


In [ ]:
from jbiophysic.tfne.tensors import compute_anisotropic_sigma
sigma = compute_anisotropic_sigma(0.3, 0.2)
print(sigma)


## 9. Optimize (Optax)


In [ ]:
def loss_fn(rates, target=10.0):
    return jnp.mean((rates - target)**2)
print("Loss defined.")


## 10. More Optimize (GSDR)


In [ ]:
from jbiophysic.optim.gsdr import gsdr_update
print("GSDR available.")


## 11. Plasticity (STDP)


In [ ]:
def stdp(dt, tp=20, tm=20, ap=0.01, am=0.012):
    return jnp.where(dt > 0, ap * jnp.exp(-dt/tp), -am * jnp.exp(dt/tm))


## 12. Neurotransmission


In [ ]:
def syn_step(s, spk, tau=5.0, dt=0.1):
    return s + dt * (-s/tau + spk*(1-s))


---
**Verification Status:** tutorial_exploratory_not_biological_truth
